In [33]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
import numpy as np
import os
os.environ['WANDB_DISABLED'] = 'true'
from itertools import combinations
from collections import Counter

# set directories
WD_junxi = Path('PATH_TO_DATA')
WD = WD_junxi
data_dir = Path(WD/'EntTemplates/Analysis/python_Patent/data/')
output_dir = Path(WD/'EntTemplates/Analysis/python_Patent/output/')
# Ensure prediction output directory exists
pred_output_dir = output_dir / "patent_family_predictions"
pred_output_dir.mkdir(parents=True, exist_ok=True)
patent_model_dir = Path('PATH_TO_PATENT_MODEL_FOLDERS/')

## Training (old, see _win for latest)

In [25]:
df = pd.read_parquet(data_dir/'all_abstracts_titles_combined.parquet')
# only keep APPLN_ID, APPLN_ABSTRACT
df = df[['APPLN_ID', 'APPLN_ABSTRACT','APPLN_ABSTRACT_LG']]
# rename to appid and abstract
df = df.rename(columns={'APPLN_ID': 'appid', 'APPLN_ABSTRACT': 'abstract', 'APPLN_ABSTRACT_LG': 'abstract_language'})
df_info = pd.read_csv(data_dir/'patent_family_mapping_starting_from_earliest_publication_year_2010.csv')
# rename APPLN_ID to appid
df_info = df_info.rename(columns={'APPLN_ID': 'appid'})
# merge to get only those in patent families
df_merged = pd.merge(df, df_info, on='appid', how='outer', indicator=True)
del df
del df_info

In [28]:
df_merged.shape

(5053476, 30)

In [9]:
# keep if GRANTED == Y
df_merged = df_merged[df_merged['GRANTED'] == 'Y'].reset_index(drop=True)
abstract_clean = df_merged['abstract'].fillna('').astype(str).str.strip()
valid_abstract = abstract_clean != ''
us_valid = (df_merged['APPLN_AUTH'] == 'US') & valid_abstract
en_valid = valid_abstract & (df_merged['abstract_language'] == 'en')

first_us_idx = (
    df_merged[us_valid]
    .reset_index()
    .groupby('DOCDB_FAMILY_ID')['index']
    .first()
)

first_en_idx = (
    df_merged[en_valid]
    .reset_index()
    .groupby('DOCDB_FAMILY_ID')['index']
    .first()
)

primary_indices = set(first_us_idx.values.tolist())
fallback_families = first_en_idx.index.difference(first_us_idx.index)
primary_indices.update(first_en_idx.loc[fallback_families].tolist())

df_merged['primary_en'] = np.where(df_merged.index.isin(primary_indices), 'Y', 'N')

df_to_process = df_merged[df_merged['primary_en'] == 'Y'].reset_index(drop=True)
print(f"Number of patent families to process: {df_to_process['DOCDB_FAMILY_ID'].nunique()}")
# also print number of families without any english abstracts and total number of families
total_families = df_merged['DOCDB_FAMILY_ID'].nunique()
families_without_en = df_merged[~df_merged['DOCDB_FAMILY_ID'].isin(first_en_idx.index)]['DOCDB_FAMILY_ID'].nunique()
print(f"Total number of patent families: {total_families}")
print(f"Number of families without any English abstracts: {families_without_en}")

Number of patent families to process: 1168626
Total number of patent families: 1308381
Number of families without any English abstracts: 139755


In [ ]:
# Clean and prepare prediction data to mirror train_predict_pat01 logic
# Keep only the needed columns to avoid unsupported pandas dtypes (e.g., categoricals)
df_pred = df_to_process[['appid', 'abstract']].copy()

# sampling 10000 for testing
df_pred = df_pred.sample(n=10000, random_state=42).reset_index(drop=True)
df_pred['appid'] = df_pred['appid'].astype(str)
df_pred = df_pred[df_pred['abstract'].notna()].copy()
df_pred['abstract'] = df_pred['abstract'].fillna('').astype(str)
df_pred = df_pred[df_pred['abstract'].str.strip() != '']

# Discover all subsegment model folders
model_base = patent_model_dir
model_folders = [p for p in model_base.glob("SUBSEGMENT-*") if (p / "best_model" / "model.safetensors").exists()]
# sort by subsegment name for consistent processing order
model_folders = sorted(model_folders, key=lambda p: p.name)
print(f"Discovered {len(model_folders)} subsegment model folders.")
results_collection = []

# Use the original base tokenizer (not saved in best_model)
tokenizer_base = "bert-base-uncased"

for model_root in model_folders:
    subsegment = model_root.name.replace("SUBSEGMENT-", "")
    out_path = pred_output_dir / f"{subsegment}_patent_family_predictions.csv"

    # Skip if results already exist
    if out_path.exists():
        print(f"Skipping {subsegment}: output already exists at {out_path.name}")
        continue

    print(f"Processing subsegment: {subsegment}")
    model_path = model_root / "best_model"  # weights live here

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_base)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    trainer = Trainer(model=model)

    hf_ds = Dataset.from_pandas(df_pred, preserve_index=False)

    def safe_tokenize_function(examples):
        return tokenizer(examples['abstract'], padding="max_length", truncation=True, max_length=512)

    predict_dataset = hf_ds.map(safe_tokenize_function, batched=True, remove_columns=hf_ds.column_names)

    predictions = trainer.predict(predict_dataset)
    logits = predictions.predictions
    y_neg = logits[:, 0]
    y_pos = logits[:, 1]
    y_pred = np.argmax(logits, axis=-1)

    df_results = pd.DataFrame({
        'appid': df_pred['appid'],
        'abstract': df_pred['abstract'],
        'pred_pos': y_pos,
        'pred_neg': y_neg,
        'positive': y_pred,
        'sector': subsegment
    })

    df_results.to_csv(out_path, index=False)
    results_collection.append((subsegment, df_results.head()))

# results_collection now holds (subsegment, preview_df) tuples for quick inspection
results_collection

## Post processing

### Combine all sector prediction results

In [34]:
## Compile all positive results
all_results = []
for file in pred_output_dir.glob("*_patent_family_predictions_50k.csv"):
    # only read in appid, posititve, sector columns
    df_part = pd.read_csv(file, usecols=['appid', 'positive', 'sector'])
    # reset index and call previous index patent_id
    df_part = df_part.reset_index()
    df_part = df_part[df_part['positive'] == 1]
    all_results.append(df_part)
df = pd.concat(all_results, ignore_index=True)
### GET TRAINING DATA
import re
# read in patent data
g_patent = pd.read_csv(data_dir/'g_patent.tsv', sep='\t', low_memory=False) 
g_patent = g_patent[g_patent['patent_type'] == 'utility']

g_assignee = pd.read_csv(data_dir/'g_assignee_disambiguated.tsv', sep='\t', low_memory=False)
g_assignee = g_assignee[g_assignee['assignee_sequence'] == 0]
g_assignee = g_assignee[['patent_id', 'assignee_id', 'disambig_assignee_organization', 'location_id']]
g_patent = g_patent.merge(g_assignee, on='patent_id', how='inner')
del g_assignee

g_location = pd.read_csv(data_dir/'g_location_disambiguated.tsv', sep='\t', low_memory=False)
g_patent = g_patent.merge(g_location[['location_id', 'disambig_country']], on='location_id', how='left')
pb_assignee = pd.read_excel(data_dir/'matched_result.xlsx')

pb_marketmap = pd.read_csv(data_dir/'predicted_positive_v2.csv')
pb_marketmap.sort_values(by=['companyid'], inplace=True)

pb_marketmap['count_sector'] = pb_marketmap.groupby('companyid')['companyid'].transform('count')
pb_marketmap['unique_segment'] = pb_marketmap['marketmap'] + pb_marketmap['segment']
pb_marketmap['count_segment'] = pb_marketmap.groupby('companyid')['unique_segment'].transform('nunique')
pb_marketmap['count_marketmap'] = pb_marketmap.groupby('companyid')['marketmap'].transform('nunique')

pb_marketmap = pb_marketmap[pb_marketmap['count_marketmap'] == 1]

pb_marketmap = pd.merge(pb_marketmap, pb_assignee[['companyid', 'assignee_id']], on='companyid', how='inner')

df_training = pd.merge(g_patent, pb_marketmap, on='assignee_id', how='inner')

def subsegment_name_processing(nameStr):
    nameStr = re.sub(r'[^\w\s]|_', '', nameStr)
    return nameStr.replace(" ", "")
df_training['file'] = df_training['fullname'].apply(subsegment_name_processing)
count_df_training = df_training.groupby('file').size().reset_index(name='counts_training_data')
# rename file to sector
count_df_training = count_df_training.rename(columns={'file':'sector'})
count_df_results = df.groupby('sector').size().reset_index(name='counts_results')
count_df_results_and_training = count_df_results.merge(count_df_training, how='outer', on='sector')
poor_sectors_df = count_df_results_and_training[(count_df_results_and_training['counts_training_data'].isnull()) | (count_df_results_and_training['counts_training_data'] < 20)]
poor_sectors_list = poor_sectors_df['sector'].unique().tolist()
# filter out poor sectors in df_results 
df_results = df[~df['sector'].isin(poor_sectors_list)]
# merge to get only those in patent families
df_text = pd.read_parquet(data_dir/'all_abstracts_titles_combined.parquet')
# only keep APPLN_ID, APPLN_ABSTRACT
df_text = df_text[['APPLN_ID', 'APPLN_ABSTRACT','APPLN_ABSTRACT_LG']]
# rename to appid and abstract
df_text = df_text.rename(columns={'APPLN_ID': 'appid', 'APPLN_ABSTRACT': 'abstract', 'APPLN_ABSTRACT_LG': 'abstract_language'})
df_info = pd.read_csv(data_dir/'patent_family_mapping_starting_from_earliest_publication_year_2010.csv')
# rename APPLN_ID to appid
df_info = df_info.rename(columns={'APPLN_ID': 'appid'})
df_merged = pd.merge(df_text, df_info, on='appid', how='inner')
print(f"Number of patent families to process after filtering: {df_merged['DOCDB_FAMILY_ID'].nunique()}; number of patents: {df_merged.shape[0]}; number of sectors: {df_results['sector'].nunique()}")

Number of patent families to process after filtering: 1751234; number of patents: 4884356; number of sectors: 172


In [35]:

df_results_with_family = pd.merge(df_results[['appid','sector']], df_merged[['appid', 'DOCDB_FAMILY_ID']], on='appid', how='inner')

### Generating filtered patent family data

In [36]:
import pycountry

## Filtering desired patents
df_merged_filter = df_merged[
    [
        "appid",
        "abstract",
        "abstract_language",
        "APPLN_AUTH",
        "APPLN_NR",
        "APPLN_KIND",
        "APPLN_FILING_DATE",
        "APPLN_FILING_YEAR",
        "GRANTED",
        "DOCDB_FAMILY_ID",
    ]
].copy()

def _print_counts(step, before_df, after_df):
    print(
        f"{step}\n"
        f"  before: unique appid={before_df['appid'].nunique():,}, unique DOCDB_FAMILY_ID={before_df['DOCDB_FAMILY_ID'].nunique():,}\n"
        f"  after : unique appid={after_df['appid'].nunique():,}, unique DOCDB_FAMILY_ID={after_df['DOCDB_FAMILY_ID'].nunique():,}\n"
    )


# Filter step 1: keep only GRANTED == 'Y'
_before = df_merged_filter
df_merged_filter = df_merged_filter[df_merged_filter["GRANTED"] == "Y"].reset_index(drop=True)
_print_counts("Filter: GRANTED == 'Y'", _before, df_merged_filter)

# Ensure APPLN_FILING_DATE is datetime (not a filter step)
df_merged_filter["APPLN_FILING_DATE_dt"] = pd.to_datetime(
    df_merged_filter["APPLN_FILING_DATE"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Earliest filing date within each family (not a filter step)
df_merged_filter["earliest_filing_date"] = (
    df_merged_filter.groupby("DOCDB_FAMILY_ID")["APPLN_FILING_DATE_dt"].transform("min")
)

# Filter step 2: keep rows within 48 months of earliest filing date
upper_bound = df_merged_filter["earliest_filing_date"] + pd.DateOffset(months=48)
in_window = (
    df_merged_filter["APPLN_FILING_DATE_dt"].notna()
    & df_merged_filter["earliest_filing_date"].notna()
    & (df_merged_filter["APPLN_FILING_DATE_dt"] >= df_merged_filter["earliest_filing_date"])
    & (df_merged_filter["APPLN_FILING_DATE_dt"] <= upper_bound)
)

df_merged_filter["within_48_months"] = np.where(in_window, "Y", "N")

print(
    f"Rows within 48 months: {(df_merged_filter['within_48_months'] == 'Y').sum():,}\n"
    f"Rows outside 48 months: {(df_merged_filter['within_48_months'] == 'N').sum():,}"
)

_before = df_merged_filter
df_merged_filter = df_merged_filter[df_merged_filter["within_48_months"] == "Y"].reset_index(drop=True)
_print_counts("Filter: within_48_months == 'Y'", _before, df_merged_filter)

# Filter step 3: drop duplicates in (DOCDB_FAMILY_ID, APPLN_AUTH)
_before = df_merged_filter
df_merged_filter = df_merged_filter.drop_duplicates(subset=["DOCDB_FAMILY_ID", "APPLN_AUTH"]).reset_index(drop=True)
_print_counts("Filter: drop_duplicates(['DOCDB_FAMILY_ID', 'APPLN_AUTH'])", _before, df_merged_filter)

# Filter step 4: drop if APPLN_AUTH is WO
_before = df_merged_filter
df_merged_filter = df_merged_filter[(df_merged_filter["APPLN_AUTH"] != "WO")].reset_index(drop=True)
_print_counts("Filter: APPLN_AUTH != 'WO'", _before, df_merged_filter)

# drop the date columns used for filtering
df_merged_filter = df_merged_filter.drop(columns=["APPLN_FILING_DATE_dt", "earliest_filing_date"])

# sort by DOCDB_FAMILY_ID and appid
df_merged_filter.sort_values(by=['DOCDB_FAMILY_ID','appid',], inplace=True)

def code_to_country(alpha2):
    if not isinstance(alpha2, str):
        return None
    country = pycountry.countries.get(alpha_2=alpha2.upper())
    return country.name if country else None


Filter: GRANTED == 'Y'
  before: unique appid=4,884,356, unique DOCDB_FAMILY_ID=1,751,234
  after : unique appid=2,943,750, unique DOCDB_FAMILY_ID=1,282,677

Rows within 48 months: 2,767,374
Rows outside 48 months: 176,376
Filter: within_48_months == 'Y'
  before: unique appid=2,943,750, unique DOCDB_FAMILY_ID=1,282,677
  after : unique appid=2,767,374, unique DOCDB_FAMILY_ID=1,282,677

Filter: drop_duplicates(['DOCDB_FAMILY_ID', 'APPLN_AUTH'])
  before: unique appid=2,767,374, unique DOCDB_FAMILY_ID=1,282,677
  after : unique appid=2,539,789, unique DOCDB_FAMILY_ID=1,282,677

Filter: APPLN_AUTH != 'WO'
  before: unique appid=2,539,789, unique DOCDB_FAMILY_ID=1,282,677
  after : unique appid=2,529,165, unique DOCDB_FAMILY_ID=1,282,666



### Spot checks

In [34]:


# Use df_merged_filter which is already at appid-DOCDB_FAMILY_ID-country level
family_countries = (
    df_merged_filter
    .dropna(subset=["country", "DOCDB_FAMILY_ID"])
    .groupby("DOCDB_FAMILY_ID")["country"]
    .apply(lambda s: sorted(set(s)))
)

pair_counter = Counter()
for countries in family_countries:
    if len(countries) < 2:
        continue
    pair_counter.update(combinations(countries, 2))

df_country_pairs = (
    pd.DataFrame(pair_counter.items(), columns=["country_pair", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

# Split country_pair into two columns for readability
df_country_pairs[["country_1", "country_2"]] = pd.DataFrame(df_country_pairs["country_pair"].tolist(), index=df_country_pairs.index)
df_country_pairs = df_country_pairs[["country_1", "country_2", "count"]]
print(df_country_pairs.head(20))



   country_1 country_2   count
0         CN        US  146600
1         JP        US   85485
2         KR        US   80104
3         CN        KR   70503
4         CA        US   63994
5         CN        JP   62367
6         ES        US   53784
7         AU        US   51565
8         AT        US   49846
9         CN        ES   39607
10        TW        US   39111
11        AU        CA   38156
12        CA        CN   38120
13        JP        KR   35825
14        AT        ES   34873
15        AU        CN   34811
16        AT        CN   32799
17        CN        TW   32231
18        CA        ES   29707
19        RU        US   29271


In [41]:
core_countries = {
    "AU", "AT", "BE", "CA", "DK", "FI", "FR", "DE", "GR", "IS", "IE",
    "IT", "JP", "LU", "NL", "NZ", "NO", "PT", "ES", "SE", "CH", "TR", "GB","US"
}


# Use df_merged_filter which is already at appid-DOCDB_FAMILY_ID-country level
family_countries_not_core = (
    df_merged_filter[~df_merged_filter['APPLN_AUTH'].isin(core_countries)]
    .dropna(subset=["country", "DOCDB_FAMILY_ID"])
    .groupby("DOCDB_FAMILY_ID")["country"]
    .apply(lambda s: sorted(set(s)))
)

pair_counter = Counter()
for countries in family_countries_not_core:
    if len(countries) < 2:
        continue
    pair_counter.update(combinations(countries, 2))

df_country_pairs = (
    pd.DataFrame(pair_counter.items(), columns=["country_pair", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

# Split country_pair into two columns for readability
df_country_pairs[["country_1", "country_2"]] = pd.DataFrame(df_country_pairs["country_pair"].tolist(), index=df_country_pairs.index)
df_country_pairs = df_country_pairs[["country_1", "country_2", "count"]]
print(df_country_pairs.head(20))

   country_1           country_2  count
0      China               Korea  70503
1      China              Taiwan  32231
2      China  Russian Federation  28437
3      Korea              Taiwan  28345
4      China              Mexico  23527
5     Brazil               China  19283
6      China              Poland  18367
7      Korea              Mexico  16593
8      Korea  Russian Federation  16550
9      China        South Africa  14896
10    Mexico  Russian Federation  12563
11    Brazil               Korea  12314
12     Korea              Poland  11373
13    Mexico        South Africa  10869
14    Brazil              Mexico  10725
15    Mexico              Poland  10165
16     China              Israel   9796
17    Poland  Russian Federation   9789
18    Brazil  Russian Federation   9687
19     Korea        South Africa   9687


### Assign sectors

In [37]:
## Assign predicted sectors to all family members
sectors_by_family = (
    df_results_with_family[['DOCDB_FAMILY_ID', 'sector']]
    .dropna(subset=['DOCDB_FAMILY_ID', 'sector'])
    .drop_duplicates()
    .reset_index(drop=True)
)
sectors_by_family

df_merged_with_sectors = df_merged_filter.merge(sectors_by_family, on='DOCDB_FAMILY_ID', how='inner')
df_merged_with_sectors.sort_values(by=['DOCDB_FAMILY_ID','appid',], inplace=True)
df_merged_with_sectors.to_csv(output_dir/'positive_results_patent_family_sector_50k.csv', index=False)

### turning it into country-macro sector level data

In [38]:
# Country–country co-occurrence counts within each sector (per patent family)
# 1) Unique country list per (sector, DOCDB_FAMILY_ID)
family_sector_countries = (
    df_merged_with_sectors
    .dropna(subset=["APPLN_AUTH", "DOCDB_FAMILY_ID", "sector"])
    .groupby(["sector", "DOCDB_FAMILY_ID"])["APPLN_AUTH"]
    .apply(lambda s: sorted(set(s)))
)

# 2) Count unordered country pairs across families within each sector
pair_counter = Counter()
for (sector, family_id), countries in family_sector_countries.items():
    if len(countries) < 2:
        continue
    for pair in combinations(countries, 2):
        pair_counter[(sector, pair[0], pair[1])] += 1

# 3) Build pair DataFrame
if pair_counter:
    df_country_pairs_by_sector = (
        pd.DataFrame(
            [(k[0], k[1], k[2], v) for k, v in pair_counter.items()],
            columns=["sector", "country_1", "country_2", "count"]
        )
        .sort_values(["sector", "count"], ascending=[True, False])
        .reset_index(drop=True)
    )

    # Total transfers per country within each sector (sum of counts across all partners)
    country_transfer_counts = (
        df_country_pairs_by_sector[["sector", "country_1", "country_2", "count"]]
        .melt(id_vars=["sector", "count"], value_vars=["country_1", "country_2"], value_name="APPLN_AUTH")
        .groupby(["sector", "APPLN_AUTH"])["count"]
        .sum()
        .rename("count_transfers_sector_worldwide")
        .reset_index()
    )

    df_country_pairs_by_sector = (
        df_country_pairs_by_sector
        .merge(
            country_transfer_counts,
            left_on=["sector", "country_1"],
            right_on=["sector", "APPLN_AUTH"],
            how="left"
        )
        .rename(columns={"count_transfers_sector_worldwide": "country_1_transfers_sector_world"})
        .drop(columns=["APPLN_AUTH"])
        .merge(
            country_transfer_counts,
            left_on=["sector", "country_2"],
            right_on=["sector", "APPLN_AUTH"],
            how="left"
        )
        .rename(columns={"count_transfers_sector_worldwide": "country_2_transfers_sector_world"})
        .drop(columns=["APPLN_AUTH"])
    )
else:
    df_country_pairs_by_sector = pd.DataFrame(
        columns=[
            "sector",
            "country_1",
            "country_2",
            "count",
            "country_1_count_transfers_sector_worldwide",
            "country_2_count_transfers_sector_worldwide",
        ]
    )

df_country_pairs_by_sector.to_stata(output_dir/'patent_transfer_country_pair_by_sector.dta', write_index=False)

### Worldwide transfer pairs

In [39]:
df_country_pairs_by_sector = pd.read_stata(output_dir/'patent_transfer_country_pair_by_sector.dta')

In [40]:
epo_share = pd.read_excel(data_dir/'2013_EPO_granted_patents_en.xlsx')
epo_countries = [
    "AT", "BE", "BG", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR",
    "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PT", "RO", "SK", "SI", "SE", "GB"
]
epo_share = epo_share[epo_share['country_2digit'].isin(epo_countries)]

In [41]:
# Worldwide transfer pairs: create non-EPO and EPO-redistributed versions
blocked_epo_partners = set(["EP"]) | set(epo_share['country_2digit'].dropna().unique())

# Ensure country pairs are consistently ordered (country_1 <= country_2) so merges aggregate correctly
df_pairs = df_country_pairs_by_sector.copy()
_swap_mask_base = df_pairs['country_1'] > df_pairs['country_2']
df_pairs.loc[_swap_mask_base, ['country_1', 'country_2']] = df_pairs.loc[_swap_mask_base, ['country_2', 'country_1']].values
df_pairs.loc[_swap_mask_base, ['country_1_transfers_sector_world', 'country_2_transfers_sector_world']] = df_pairs.loc[_swap_mask_base, ['country_2_transfers_sector_world', 'country_1_transfers_sector_world']].values

# 1) Drop pairs involving EP or any EPO member country
worldwide_pairs_filtered = df_pairs[
    (~df_pairs['country_1'].isin(blocked_epo_partners))
    & (~df_pairs['country_2'].isin(blocked_epo_partners))
].copy()
worldwide_pairs_filtered['fullname_raw'] = worldwide_pairs_filtered['sector'].str.lower()
worldwide_pairs_filtered.to_stata(
    output_dir/'worldwide_pairs_filtered_no_EPO_countries.dta',
    write_index=False,
)

# 2) Redistribute EP counts to member countries using EPO patent shares
_epo_weights = epo_share[['country_2digit', 'share_patent']].dropna().copy()
_epo_weights['share_patent_norm'] = _epo_weights['share_patent'] / _epo_weights['share_patent'].sum()

_ep_rows_mask = (df_pairs['country_1'] == 'EP') | (df_pairs['country_2'] == 'EP')
_ep_rows = df_pairs[_ep_rows_mask]

redistributed_rows = []
if not _epo_weights.empty and not _ep_rows.empty:
    for _, r in _ep_rows.iterrows():
        for _, w in _epo_weights.iterrows():
            weight = w['share_patent_norm']
            new_row = r.copy()
            # Replace EP side with member country and scale partner totals
            if r['country_1'] == 'EP':
                new_row['country_1'] = w['country_2digit']
                new_row['country_1_transfers_sector_world'] = (
                    r['country_1_transfers_sector_world'] * weight
                )
            if r['country_2'] == 'EP':
                new_row['country_2'] = w['country_2digit']
                new_row['country_2_transfers_sector_world'] = (
                    r['country_2_transfers_sector_world'] * weight
                )
            new_row['count'] = r['count'] * weight
            redistributed_rows.append(new_row)

non_ep_rows = df_pairs[~_ep_rows_mask]
worldwide_pairs_redistributed = pd.concat(
    [non_ep_rows, pd.DataFrame(redistributed_rows)], ignore_index=True
)

# Normalize ordering again after redistribution so duplicates collapse on groupby
_swap_mask_redist = worldwide_pairs_redistributed['country_1'] > worldwide_pairs_redistributed['country_2']
worldwide_pairs_redistributed.loc[_swap_mask_redist, ['country_1', 'country_2']] = worldwide_pairs_redistributed.loc[_swap_mask_redist, ['country_2', 'country_1']].values
worldwide_pairs_redistributed.loc[_swap_mask_redist, ['country_1_transfers_sector_world', 'country_2_transfers_sector_world']] = worldwide_pairs_redistributed.loc[_swap_mask_redist, ['country_2_transfers_sector_world', 'country_1_transfers_sector_world']].values

agg_spec_world = {
    'count': 'sum',
    'country_1_transfers_sector_world': 'sum',
    'country_2_transfers_sector_world': 'sum',
}
worldwide_pairs_redistributed = (
    worldwide_pairs_redistributed
    .groupby(['sector', 'country_1', 'country_2'], as_index=False)
    .agg(agg_spec_world)
)
# generate fullname_raw
worldwide_pairs_redistributed['fullname_raw'] = worldwide_pairs_redistributed['sector'].str.lower()

worldwide_pairs_redistributed.to_stata(
    output_dir/'worldwide_pairs_filtered_EPO_redistributed.dta',
    write_index=False,
)

### Transferring with Chinese

In [42]:
epo_share = pd.read_excel(data_dir/'2013_EPO_granted_patents_en.xlsx')
epo_countries = [
    "AT", "BE", "BG", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR",
    "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PT", "RO", "SK", "SI", "SE", "GB"
]
epo_share = epo_share[epo_share['country_2digit'].isin(epo_countries)]

In [43]:
# Keep only pairs involving China and collapse to country-level (China vs. partner)
cn_pairs = df_country_pairs_by_sector[
    (df_country_pairs_by_sector["country_1"] == "CN")
    | (df_country_pairs_by_sector["country_2"] == "CN")
].copy()

cn_pairs["partner_country"] = np.where(
    cn_pairs["country_1"] == "CN", cn_pairs["country_2"], cn_pairs["country_1"]
)
cn_pairs["partner_transfers_sector_world"] = np.where(
    cn_pairs["country_1"] == "CN",
    cn_pairs["country_2_transfers_sector_world"],
    cn_pairs["country_1_transfers_sector_world"],
)
cn_pairs["china_transfers_sector_world"] = np.where(
    cn_pairs["country_1"] == "CN",
    cn_pairs["country_1_transfers_sector_world"],
    cn_pairs["country_2_transfers_sector_world"],
)
cn_pairs = cn_pairs[['sector', 'partner_country', 'count', 'china_transfers_sector_world', 'partner_transfers_sector_world']].copy()

# 2) Filtered version: drop EP and all partners in epo_share list
blocked_partners = set(['EP']) | set(epo_share['country_2digit'].dropna().unique())
cn_pairs_filtered = cn_pairs[~cn_pairs['partner_country'].isin(blocked_partners)].copy()
# rename partner_country to country_2digit, rename sector to fullname_raw, make fullname_raw lowercase
cn_pairs_filtered = cn_pairs_filtered.rename(columns={'partner_country': 'country_2digit', 'sector': 'fullname_raw'})
cn_pairs_filtered['fullname_raw'] = cn_pairs_filtered['fullname_raw'].str.lower()
cn_pairs_filtered.to_stata(output_dir/'cn_pairs_filtered_no_EPO_countries.dta', write_index=False)

# 3) Redistribute EP counts to member countries using EPO patent shares
# Normalize weights from epo_share
_epo_weights = epo_share[['country_2digit', 'share_patent']].dropna().copy()
_epo_weights['share_patent_norm'] = _epo_weights['share_patent'] / _epo_weights['share_patent'].sum()

_ep_rows = cn_pairs[cn_pairs['partner_country'] == 'EP']
if not _epo_weights.empty and not _ep_rows.empty:
    redistributed_rows = []
    for _, r in _ep_rows.iterrows():
        for _, w in _epo_weights.iterrows():
            weight = w['share_patent_norm']
            new_row = r.copy()
            new_row['partner_country'] = w['country_2digit']
            new_row['count'] = r['count'] * weight
            # scale partner-side totals to match redistributed counts
            new_row['partner_transfers_sector_world'] = r['partner_transfers_sector_world'] * weight
            redistributed_rows.append(new_row)
    cn_pairs_redistributed = pd.concat(
        [cn_pairs[cn_pairs['partner_country'] != 'EP'], pd.DataFrame(redistributed_rows)],
        ignore_index=True,
    )
else:
    cn_pairs_redistributed = cn_pairs.copy()

# Aggregate so redistributed EP weight is added to any existing partner rows
agg_spec = {
    'count': 'sum',
    'partner_transfers_sector_world': 'sum',
    # keep China's total once per (sector, partner_country)
    'china_transfers_sector_world': 'first',
}
cn_pairs_redistributed = (
    cn_pairs_redistributed
    .groupby(['sector', 'partner_country'], as_index=False)
    .agg(agg_spec)
)
# rename partner_country to country_2digit, rename sector to fullname_raw, make fullname_raw lowercase
cn_pairs_redistributed = cn_pairs_redistributed.rename(columns={'partner_country': 'country_2digit', 'sector': 'fullname_raw'})
cn_pairs_redistributed['fullname_raw'] = cn_pairs_redistributed['fullname_raw'].str.lower()
cn_pairs_redistributed.to_stata(output_dir/'cn_pairs_EPO_redistributed.dta', write_index=False)

### Transferring with US

In [44]:
# Keep only pairs involving the US and collapse to country-level (US vs. partner)
us_pairs = df_country_pairs_by_sector[
    (df_country_pairs_by_sector["country_1"] == "US")
    | (df_country_pairs_by_sector["country_2"] == "US")
].copy()

us_pairs["partner_country"] = np.where(
    us_pairs["country_1"] == "US", us_pairs["country_2"], us_pairs["country_1"]
)
us_pairs["partner_transfers_sector_world"] = np.where(
    us_pairs["country_1"] == "US",
    us_pairs["country_2_transfers_sector_world"],
    us_pairs["country_1_transfers_sector_world"],
)
us_pairs["us_transfers_sector_world"] = np.where(
    us_pairs["country_1"] == "US",
    us_pairs["country_1_transfers_sector_world"],
    us_pairs["country_2_transfers_sector_world"],
)
us_pairs = us_pairs[['sector', 'partner_country', 'count', 'us_transfers_sector_world', 'partner_transfers_sector_world']].copy()

# Filtered version for US: drop EP and all partners in epo_share list
blocked_partners_us = set(['EP']) | set(epo_share['country_2digit'].dropna().unique())
us_pairs_filtered = us_pairs[~us_pairs['partner_country'].isin(blocked_partners_us)].copy()

# rename partner_country to country_2digit, rename sector to fullname_raw, make fullname_raw lowercase
us_pairs_filtered = us_pairs_filtered.rename(columns={'partner_country': 'country_2digit', 'sector': 'fullname_raw'})
us_pairs_filtered['fullname_raw'] = us_pairs_filtered['fullname_raw'].str.lower()
us_pairs_filtered.to_stata(output_dir/'us_pairs_filtered_no_EPO_countries.dta', write_index=False)


# Redistribute EP counts in US pairs to member countries using EPO patent shares
_epo_weights_us = epo_share[['country_2digit', 'share_patent']].dropna().copy()
_epo_weights_us['share_patent_norm'] = _epo_weights_us['share_patent'] / _epo_weights_us['share_patent'].sum()

_ep_rows_us = us_pairs[us_pairs['partner_country'] == 'EP']
if not _epo_weights_us.empty and not _ep_rows_us.empty:
    redistributed_rows_us = []
    for _, r in _ep_rows_us.iterrows():
        for _, w in _epo_weights_us.iterrows():
            weight = w['share_patent_norm']
            new_row = r.copy()
            new_row['partner_country'] = w['country_2digit']
            new_row['count'] = r['count'] * weight
            new_row['partner_transfers_sector_world'] = r['partner_transfers_sector_world'] * weight
            redistributed_rows_us.append(new_row)
    us_pairs_redistributed = pd.concat(
        [us_pairs[us_pairs['partner_country'] != 'EP'], pd.DataFrame(redistributed_rows_us)],
        ignore_index=True,
    )
else:
    us_pairs_redistributed = us_pairs.copy()

agg_spec_us = {
    'count': 'sum',
    'partner_transfers_sector_world': 'sum',
    'us_transfers_sector_world': 'first',
}
us_pairs_redistributed = (
    us_pairs_redistributed
    .groupby(['sector', 'partner_country'], as_index=False)
    .agg(agg_spec_us)
)
us_pairs_redistributed = us_pairs_redistributed.rename(columns={'partner_country': 'country_2digit', 'sector': 'fullname_raw'})
us_pairs_redistributed['fullname_raw'] = us_pairs_redistributed['fullname_raw'].str.lower()

us_pairs_redistributed.to_stata(output_dir/'us_pairs_EPO_redistributed.dta', write_index=False)
